# 🚗 Server Backend YOLO + XAI + BLIP Deskripsi (Google Colab GPU)
### Sistem Analisis Investigasi Kecelakaan — Program Tesis S2

Notebook ini menjalankan backend deteksi objek YOLOv8, Grad-CAM, LIME, SHAP, dan BLIP Image Captioning berbahasa Indonesia.

**Langkah Penggunaan:**
1. Pastikan Runtime menggunakan GPU: **Runtime → Change runtime type → T4 GPU**.
2. Jalankan semua sel: **Runtime → Run all** (atau tekan `Ctrl + F9`).
3. Izinkan akses Google Drive saat diminta.
4. Tunggu hingga sel terakhir menampilkan `✅ SERVER AKTIF & SIAP MENERIMA PERMINTAAN`.

In [ ]:
# @title 1. Pemasangan Dependensi & Library Machine Learning
!pip -q install ultralytics flask flask-cors pyngrok lime shap scikit-image transformers sentencepiece sacremoses
print('✅ [1/5] Dependensi berhasil dipasang.')

In [ ]:
# @title 2. Menghubungkan Google Drive & Memuat Model
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/Program Tesis Colab')
LOCAL_MODEL_PATH = Path('/content/best.pt')

if (DRIVE_PROJECT_DIR / 'best.pt').exists():
    MODEL_PATH = DRIVE_PROJECT_DIR / 'best.pt'
    print(f'✅ [2/5] Model ditemukan di Google Drive: {MODEL_PATH}')
elif LOCAL_MODEL_PATH.exists():
    MODEL_PATH = LOCAL_MODEL_PATH
    print(f'✅ [2/5] Model ditemukan di /content/best.pt')
else:
    raise FileNotFoundError(
        '❌ File model best.pt tidak ditemukan di Drive (folder: Program Tesis Colab) maupun /content/.'
    )

In [ ]:
# @title 3. Menyiapkan Backend Server Terbaru (BLIP + XAI + YOLO)
import sys, os
from pathlib import Path

# Tulis langsung kode backend terbaru ke /content/shap_server.py
code = """\"\"\"Backend YOLO + XAI untuk dijalankan di Google Colab.\"\"\"

from __future__ import annotations

import base64
import io
import threading
import time
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from flask import Flask, jsonify, request
from flask_cors import CORS
from lime import lime_image
from PIL import Image
from skimage.segmentation import mark_boundaries
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    BlipForConditionalGeneration,
    BlipProcessor,
)
from ultralytics import YOLO


def _ke_b64(rgb: np.ndarray) -> str:
    gambar = Image.fromarray(np.uint8(np.clip(rgb, 0, 255)))
    buffer = io.BytesIO()
    gambar.save(buffer, format="JPEG", quality=92)
    return base64.b64encode(buffer.getvalue()).decode("ascii")


def _dari_b64(teks: str) -> np.ndarray:
    if "," in teks and teks.lstrip().startswith("data:"):
        teks = teks.split(",", 1)[1]
    return np.asarray(Image.open(io.BytesIO(base64.b64decode(teks))).convert("RGB"))


class MesinAnalisis:
    def __init__(self, model_path: str):
        self.yolo = YOLO(model_path)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.yolo.model.to(self.device).eval()
        self.names = self.yolo.names
        self.jumlah_kelas = len(self.names)
        self._caption_lock = threading.Lock()
        self._caption_processor = None
        self._caption_model = None
        self._translation_tokenizer = None
        self._translation_model = None

    def _muat_model_deskripsi(self) -> None:
        \"\"\"Muat model saat pertama dibutuhkan agar startup server tetap cepat.\"\"\"
        if self._caption_model is not None:
            return
        with self._caption_lock:
            if self._caption_model is not None:
                return
            caption_id = "Salesforce/blip-image-captioning-base"
            translation_id = "Helsinki-NLP/opus-mt-en-id"
            self._caption_processor = BlipProcessor.from_pretrained(caption_id)
            self._caption_model = BlipForConditionalGeneration.from_pretrained(caption_id).to(self.device).eval()
            try:
                self._translation_tokenizer = AutoTokenizer.from_pretrained(translation_id)
                self._translation_model = AutoModelForSeq2SeqLM.from_pretrained(translation_id).to(self.device).eval()
            except Exception as exc:
                self._translation_tokenizer = None
                self._translation_model = None

    def deskripsikan(self, rgb: np.ndarray) -> str:
        \"\"\"Buat caption natural dari piksel gambar lalu terjemahkan ke Indonesia.\"\"\"
        self._muat_model_deskripsi()
        gambar = Image.fromarray(np.uint8(np.clip(rgb, 0, 255)))
        with self._caption_lock, torch.inference_mode():
            masukan = self._caption_processor(images=gambar, return_tensors="pt").to(self.device)
            token_caption = self._caption_model.generate(
                **masukan,
                max_new_tokens=50,
                num_beams=4,
                repetition_penalty=1.15,
            )
            caption_en = self._caption_processor.decode(token_caption[0], skip_special_tokens=True).strip()
            deskripsi = caption_en
            if self._translation_tokenizer is not None and self._translation_model is not None:
                try:
                    token_terjemahan = self._translation_tokenizer(
                        [caption_en], return_tensors="pt", padding=True, truncation=True
                    ).to(self.device)
                    hasil = self._translation_model.generate(
                        **token_terjemahan, max_new_tokens=70, num_beams=4
                    )
                    id_trans = self._translation_tokenizer.decode(hasil[0], skip_special_tokens=True).strip()
                    if id_trans:
                        deskripsi = id_trans
                except Exception:
                    deskripsi = caption_en

        if not deskripsi:
            deskripsi = "Pemandangan kecelakaan lalu lintas."
        deskripsi = deskripsi[0].upper() + deskripsi[1:]
        return deskripsi if deskripsi.endswith((".", "!", "?")) else deskripsi + "."


    def deteksi(self, rgb: np.ndarray, confidence: float):
        mulai = time.perf_counter()
        hasil = self.yolo.predict(rgb, conf=confidence, verbose=False, device=self.device)[0]
        waktu = time.perf_counter() - mulai
        anotasi = cv2.cvtColor(hasil.plot(), cv2.COLOR_BGR2RGB)
        objek = []
        for box in hasil.boxes:
            kelas_id = int(box.cls.item())
            objek.append({
                "kelas": str(self.names[kelas_id]),
                "kelas_id": kelas_id,
                "confidence": float(box.conf.item()),
                "bbox": [float(x) for x in box.xyxy[0].tolist()],
            })
        kelas_top = max(objek, key=lambda x: x["confidence"])["kelas"] if objek else "Tidak ada"
        return hasil, anotasi, objek, kelas_top, waktu

    def _skor_batch(self, images: np.ndarray) -> np.ndarray:
        \"\"\"Skor maksimum per kelas; dipakai LIME dan SHAP.\"\"\"
        scores = np.zeros((len(images), self.jumlah_kelas), dtype=np.float32)
        for awal in range(0, len(images), 8):
            batch = [np.uint8(np.clip(x, 0, 255)) for x in images[awal:awal + 8]]
            results = self.yolo.predict(batch, conf=0.01, verbose=False, device=self.device)
            for indeks, result in enumerate(results, start=awal):
                if result.boxes is None:
                    continue
                for cls, conf in zip(result.boxes.cls.tolist(), result.boxes.conf.tolist()):
                    cls_id = int(cls)
                    scores[indeks, cls_id] = max(scores[indeks, cls_id], float(conf))
        return scores

    def gradcam(self, rgb: np.ndarray, kelas_id: int | None) -> np.ndarray:
        \"\"\"Grad-CAM pada feature layer terakhir sebelum head Detect YOLO.\"\"\"
        ukuran = 640
        resized = cv2.resize(rgb, (ukuran, ukuran))
        tensor = torch.from_numpy(resized).to(self.device).float().permute(2, 0, 1)[None] / 255.0
        tensor.requires_grad_(True)
        aktivasi: list[torch.Tensor] = []
        gradien: list[torch.Tensor] = []

        layer = self.yolo.model.model[-2]

        def simpan_aktivasi(_module, _input, output):
            value = output[0] if isinstance(output, (tuple, list)) else output
            aktivasi.append(value)
            value.register_hook(lambda grad: gradien.append(grad))

        hook = layer.register_forward_hook(simpan_aktivasi)
        try:
            with torch.enable_grad():
                self.yolo.model.zero_grad(set_to_none=True)
                raw = self.yolo.model(tensor)
                pred = raw[0] if isinstance(raw, (tuple, list)) else raw
                # Bentuk YOLOv8/YOLO11 lazim: [batch, 4 + jumlah_kelas, anchors].
                if pred.ndim != 3:
                    raise RuntimeError(f"Bentuk output YOLO tidak dikenali: {tuple(pred.shape)}")
                if pred.shape[1] >= 4 + self.jumlah_kelas:
                    skor = pred[:, 4:4 + self.jumlah_kelas, :]
                    target = skor[:, kelas_id, :].max() if kelas_id is not None else skor.max()
                else:
                    raise RuntimeError(f"Output tidak memuat {self.jumlah_kelas} skor kelas")
                target.backward()
                if not aktivasi or not gradien:
                    return rgb
                act, grad = aktivasi[-1], gradien[-1]
                bobot = grad.mean(dim=(2, 3), keepdim=True)
                cam = torch.relu((bobot * act).sum(dim=1))[0]
                cam -= cam.min()
                cam /= cam.max().clamp_min(1e-8)
                cam = cv2.resize(cam.detach().cpu().numpy(), (rgb.shape[1], rgb.shape[0]))
                warna = cv2.cvtColor(cv2.applyColorMap(np.uint8(cam * 255), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
                return np.uint8(0.55 * rgb + 0.45 * warna)
        finally:
            hook.remove()

    def lime(self, rgb: np.ndarray, kelas_id: int, samples: int) -> np.ndarray:
        kecil = cv2.resize(rgb, (320, 320))
        explainer = lime_image.LimeImageExplainer(random_state=42)
        explanation = explainer.explain_instance(
            kecil,
            classifier_fn=self._skor_batch,
            labels=(kelas_id,),
            num_samples=samples,
            hide_color=0,
        )
        temp, mask = explanation.get_image_and_mask(
            kelas_id, positive_only=False, num_features=10, hide_rest=False
        )
        visual = mark_boundaries(temp / 255.0 if temp.max() > 1 else temp, mask)
        return cv2.resize(np.uint8(np.clip(visual, 0, 1) * 255), (rgb.shape[1], rgb.shape[0]))

    def shap(self, rgb: np.ndarray, kelas_id: int, max_evals: int) -> np.ndarray:
        import shap

        kecil = cv2.resize(rgb, (128, 128))
        masker = shap.maskers.Image("blur(16,16)", kecil.shape)
        explainer = shap.Explainer(self._skor_batch, masker, output_names=list(self.names.values()))
        valores = explainer(
            kecil[None],
            max_evals=max(max_evals, 2 * 16 * 16 + 1),
            batch_size=8,
            outputs=[kelas_id],
        )
        shap.image_plot(valores, show=False)
        fig = plt.gcf()
        buffer = io.BytesIO()
        fig.savefig(buffer, format="png", bbox_inches="tight", dpi=120)
        plt.close(fig)
        buffer.seek(0)
        return np.asarray(Image.open(buffer).convert("RGB"))


def create_app(model_path: str = "/content/best.pt") -> Flask:
    app = Flask(__name__)
    app.config["MAX_CONTENT_LENGTH"] = 16 * 1024 * 1024
    CORS(app)
    mesin = MesinAnalisis(model_path)

    @app.get("/halo")
    def halo():
        return jsonify({
            "status": "sukses",
            "pesan": "Server Colab YOLO + XAI terhubung",
            "device": str(mesin.device),
            "kelas": mesin.names,
        })

    @app.post("/analisis")
    def analisis():
        try:
            payload = request.get_json(force=True)
            rgb = _dari_b64(payload["gambar"])
            confidence = float(payload.get("confidence", 0.25))
            _, anotasi, objek, kelas_top, waktu_deteksi = mesin.deteksi(rgb, confidence)
            kelas_id = max(objek, key=lambda item: item["confidence"])["kelas_id"] if objek else None
            response = {
                "status": "sukses",
                "gambar_deteksi": _ke_b64(anotasi),
                "deteksi": objek,
                "kelas_top": kelas_top,
                "waktu_deteksi": waktu_deteksi,
            }

            if payload.get("deskripsi", True):
                mulai = time.perf_counter()
                try:
                    response["deskripsi"] = mesin.deskripsikan(rgb)
                    response["waktu_deskripsi"] = time.perf_counter() - mulai
                except Exception as exc:
                    app.logger.exception("Deskripsi gambar gagal")
                    response["deskripsi"] = "Deskripsi natural belum dapat dibuat pada analisis ini."
                    response["peringatan_deskripsi"] = str(exc)

            if payload.get("gradcam", True):
                mulai = time.perf_counter()
                try:
                    response["gradcam"] = _ke_b64(mesin.gradcam(rgb, kelas_id))
                    response["waktu_gradcam"] = time.perf_counter() - mulai
                except Exception as exc:
                    app.logger.warning("Grad-CAM gagal: %s", exc)

            if payload.get("lime", False) and kelas_id is not None:
                mulai = time.perf_counter()
                try:
                    response["lime"] = _ke_b64(mesin.lime(rgb, kelas_id, int(payload.get("lime_samples", 100))))
                    response["waktu_lime"] = time.perf_counter() - mulai
                except Exception as exc:
                    app.logger.warning("LIME gagal: %s", exc)

            if payload.get("shap", False) and kelas_id is not None:
                mulai = time.perf_counter()
                try:
                    response["shap"] = _ke_b64(mesin.shap(rgb, kelas_id, int(payload.get("shap_evals", 600))))
                    response["waktu_shap"] = time.perf_counter() - mulai
                except Exception as exc:
                    app.logger.warning("SHAP gagal: %s", exc)

            return jsonify(response)
        except Exception as exc:
            app.logger.exception("Analisis gagal")
            return jsonify({"status": "gagal", "pesan": str(exc)}), 500

    return app

"""

with open('/content/shap_server.py', 'w', encoding='utf-8') as f:
    f.write(code)

# Update juga salinan di Google Drive jika terhubung
drive_server = Path('/content/drive/MyDrive/Program Tesis Colab/shap_server.py')
if drive_server.parent.exists():
    with open(drive_server, 'w', encoding='utf-8') as f:
        f.write(code)

if '/content' not in sys.path:
    sys.path.insert(0, '/content')

import shap_server
import importlib
importlib.reload(shap_server)
from shap_server import create_app
print('✅ [3/5] Modul backend terbaru (YOLO + BLIP + XAI) berhasil dimuat!')


In [ ]:
# @title 4. Konfigurasi Authtoken & Domain Ngrok
import os
from pyngrok import ngrok

token = None
try:
    from google.colab import userdata
    token = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    pass

if not token:
    token = os.environ.get('NGROK_AUTHTOKEN')

if not token:
    token = input('Masukkan NGROK_AUTHTOKEN Anda: ').strip()

assert token, '❌ NGROK_AUTHTOKEN wajib diisi!'
ngrok.set_auth_token(token)

domain = None
try:
    domain = userdata.get('NGROK_DOMAIN')
except Exception:
    pass

if domain:
    domain = domain.replace('https://', '').replace('http://', '').rstrip('/')
    print(f'✅ [4/5] Menggunakan Static Domain Ngrok: {domain}')
else:
    print('✅ [4/5] Menggunakan Dynamic Tunnel Ngrok.')

In [ ]:
# @title 5. Menjalankan Server YOLO & Membuka Akses Publik (Ngrok)
import threading, time, requests, torch
from shap_server import create_app

device_info = 'T4 GPU' if torch.cuda.is_available() else 'CPU'
print(f'🖥️  Komputasi berjalan pada: {device_info}')

app = create_app(str(MODEL_PATH))
server_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, use_reloader=False, threaded=True),
    daemon=True
)
server_thread.start()
time.sleep(3)

ngrok.kill()
try:
    if domain:
        public_url = ngrok.connect(addr=5000, bind_tls=True, domain=domain).public_url
    else:
        public_url = ngrok.connect(addr=5000, bind_tls=True).public_url
except Exception as e:
    print(f'Koneksi dengan domain khusus ({domain}) gagal ({e}), beralih ke dynamic tunnel...')
    public_url = ngrok.connect(addr=5000, bind_tls=True).public_url

health_res = requests.get(public_url + '/halo', headers={'ngrok-skip-browser-warning': '1'}, timeout=30).json()

print('=' * 75)
print('🚀  SERVER AKTIF & SIAP MENERIMA PERMINTAAN!')
print(f'🌐  URL Backend Publik : {public_url}')
print(f'📊  Device Status      : {health_res.get("device")}')
print(f'🏷️  Daftar Kelas       : {health_res.get("kelas")}')
print('=' * 75)
print('\n💡 Catatan: Biarkan notebook ini tetap terbuka selama aplikasi digunakan.\n')

while True:
    time.sleep(60)